# VQE Molecular Energy Demo with Small ESOL Candidates

This notebook introduces **VQE**, the Variational Quantum Eigensolver.

In simple words:

> VQE estimates the lowest energy of a molecule, called its ground-state energy.

This is different from QAOA:

| Algorithm | Main Use | Pharma Connection |
|---|---|---|
| QAOA | Optimization | Select the best portfolio of molecules |
| VQE | Quantum chemistry | Estimate molecular energies and stability |

We connect this demo to the ESOL dataset by selecting three very small ESOL molecules. Then we run a reduced **active-space VQE** calculation for each molecule.

Important caveat: this is not a full VQE calculation for large drug-like molecules. Full electronic-structure VQE for real drug molecules would need many more qubits than are practical today.

Architecture reference: see `../docs/architecture.md` for the full QAOA/VQE data flow and algorithm roles.


## Why Active-Space VQE?

A molecule's full quantum chemistry problem can require many spin orbitals, and each mapped spin orbital can become a qubit.

To keep the notebook runnable, we use an **active space**:

- choose a small number of electrons
- choose a small number of molecular orbitals
- freeze the rest into an effective background

This is common in quantum chemistry demos. It lets us show the VQE workflow without pretending current hardware can solve full pharma molecules.

In [ ]:
from pathlib import Path
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_SEED = 42
VQE_MAXITER = 100
ACTIVE_SPACE_ELECTRONS = 2
ACTIVE_SPACE_ORBITALS = 2

np.random.seed(RANDOM_SEED)


## 1. Load ESOL and Select Three Small Molecules

We choose small ESOL molecules because they are the only reasonable candidates for a lightweight VQE demo.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
local_paths = [Path("data/delaney-processed.csv"), Path("../data/delaney-processed.csv")]
data_path = next((path for path in local_paths if path.exists()), None)
esol = pd.read_csv(data_path if data_path else DATA_URL)

selected_names = ["Methane", "Methanol", "Ethyne"]
selected = esol[esol["Compound ID"].isin(selected_names)].copy()
selected = selected.set_index("Compound ID").loc[selected_names].reset_index()

selected[["Compound ID", "smiles", "Molecular Weight", "measured log solubility in mols per litre"]]

## 2. Approximate Molecular Geometries

VQE needs atom coordinates, not just SMILES. ESOL does not provide 3D geometries, so we use simple approximate geometries for these tiny molecules.

For serious chemistry work, geometries should come from experimental structures or proper conformer generation and geometry optimization.

In [ ]:
geometries = {
    "Methane": "C 0 0 0; H 0.629 0.629 0.629; H -0.629 -0.629 0.629; H -0.629 0.629 -0.629; H 0.629 -0.629 -0.629",
    "Methanol": "C 0 0 0; O 1.43 0 0; H -0.63 0.63 0.63; H -0.63 -0.63 0.63; H -0.63 0 -0.89; H 1.82 0.89 0",
    "Ethyne": "C 0 0 -0.60; C 0 0 0.60; H 0 0 -1.66; H 0 0 1.66",
}

for name, geometry in geometries.items():
    print(name)
    print(geometry)
    print()

## 3. Define the VQE Workflow

For each molecule:

1. Build an electronic-structure problem with PySCF.
2. Reduce it to a 2-electron, 2-orbital active space.
3. Map the fermionic Hamiltonian to qubits.
4. Run exact diagonalization as a classical reference.
5. Run VQE with a UCCSD ansatz.
6. Compare VQE energy against the exact active-space energy.

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer


def run_active_space_vqe(name, atom_geometry, basis="sto3g"):
    driver = PySCFDriver(atom=atom_geometry, basis=basis)
    full_problem = driver.run()

    active_problem = ActiveSpaceTransformer(
        num_electrons=ACTIVE_SPACE_ELECTRONS,
        num_spatial_orbitals=ACTIVE_SPACE_ORBITALS,
    ).transform(full_problem)

    mapper = ParityMapper(num_particles=active_problem.num_particles)
    qubit_operator = mapper.map(active_problem.hamiltonian.second_q_op())

    initial_state = HartreeFock(
        active_problem.num_spatial_orbitals,
        active_problem.num_particles,
        mapper,
    )
    ansatz = UCCSD(
        active_problem.num_spatial_orbitals,
        active_problem.num_particles,
        mapper,
        initial_state=initial_state,
    )

    optimizer = SLSQP(maxiter=VQE_MAXITER)
    vqe = VQE(StatevectorEstimator(), ansatz, optimizer)
    vqe_result = vqe.compute_minimum_eigenvalue(qubit_operator)
    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(qubit_operator)

    interpreted_vqe = active_problem.interpret(vqe_result)
    interpreted_exact = active_problem.interpret(exact_result)

    return {
        "compound": name,
        "full_spatial_orbitals": full_problem.num_spatial_orbitals,
        "full_particles": str(full_problem.num_particles),
        "active_spatial_orbitals": active_problem.num_spatial_orbitals,
        "active_particles": str(active_problem.num_particles),
        "active_qubits_after_mapping": qubit_operator.num_qubits,
        "pauli_terms": len(qubit_operator),
        "exact_active_energy_hartree": float(np.real(interpreted_exact.total_energies[0])),
        "vqe_active_energy_hartree": float(np.real(interpreted_vqe.total_energies[0])),
        "absolute_error_hartree": abs(
            float(np.real(interpreted_vqe.total_energies[0]))
            - float(np.real(interpreted_exact.total_energies[0]))
        ),
    }


## 4. Run VQE for the Three ESOL Candidates

These are active-space calculations. The results are useful for learning the workflow, not for ranking the molecules as drug candidates.

In [ ]:
results = [run_active_space_vqe(name, geometry) for name, geometry in geometries.items()]
results_df = pd.DataFrame(results)
results_df

## 5. Compare Exact and VQE Active-Space Energies

VQE should closely match the exact active-space reference for these tiny reduced problems.

In [ ]:
plot_df = results_df.set_index("compound")[["exact_active_energy_hartree", "vqe_active_energy_hartree"]]

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_title("Exact vs VQE Active-Space Energies")
ax.set_ylabel("Energy (Hartree)")
ax.set_xlabel("ESOL candidate")
ax.legend(["Exact active-space", "VQE active-space"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

results_df[["compound", "absolute_error_hartree", "active_qubits_after_mapping", "pauli_terms"]]

## 6. Interpretation

What this notebook shows:

- VQE estimates molecular ground-state energies.
- VQE is closer to quantum chemistry than QAOA.
- We can connect ESOL candidates to VQE only for very small molecules or reduced active spaces.
- The active-space VQE result can match exact diagonalization very closely for these tiny examples.

What this notebook does not claim:

- It does not compute full production-quality energies for drug-like molecules.
- It does not use ESOL solubility as a VQE target.
- It does not prove quantum advantage.

A realistic future extension would use better 3D conformers, geometry optimization, larger active spaces, and careful comparison to classical quantum chemistry methods.